In [4]:
"""
Created on Mon Mar 10 2025
@author: Max Van Migem
"""
import numpy as np
import os
import time
import sys
import pickle
# from psychopy import parallel, visual, gui, data, event, core, monitors
# from psychopy.visual import ShapeStim
# from psychopy import logging
from math import fabs
rng = np.random.default_rng(seed=None)
linejitter_arr =(np.ones((30,30,2))- rng.random((30,30,2))*2)/80
# file_name = os.getcwd() + '/stim_jitter.npy'
# np.save(file_name,linejitter_arr)

In [2]:
n_trials = 28
n_odd = 7
n_target = 9
n_blocks = 32
avoid_quandrant = 3

isi_duration = .52
stim_onset_jitter = .07
iti_duration = .5
stim_duration = .1
subject_code = [0,0,1,0] 

In [3]:
def generateTargetTrials(tr_block, ntarget):
    """ 
    Some trials are target trials, this function generates a list to designate which trials are
    Parameters:
        tr_block (int):number of trials per block
        ntarget (int): number of target trials
    Return:
        full_list (list of ints): list of targets for every trial in a block
    """
    # Make a list with normal trials (0) and a list with target trials (1)
    full_list = np.zeros(tr_block-ntarget,dtype=int)
    c_list = np.ones(ntarget*3,dtype=int) # make a longer list and cut it short otherwise it won't work with ntarget <3
    for i,one in enumerate(c_list):
        c_list[i] = int(one + i%3)   # Just adding 0,1, or 2  to every element (again: if ntarget is divisible by 3 then it's balanced)
    # Make correct length
    np.random.shuffle(c_list)
    c_list = c_list[:ntarget]

    # Make one list
    full_list = np.append(full_list,c_list)
    # Shuffle
    np.random.shuffle(full_list)

    return full_list

def generateTrialTimings(tr_block,isi_dur, jitter,randng=rng):
    """ 
    Generate a list of timings 4 per trial
    Parameters:
        tr_block (int):number of trials per block
        isi_dur (float): mean duration of isi
        jitter (float): variation on isi duration
        randng (np Generator class): random number generator
    """
    block_tlist= []
    for i in range(tr_block):
        trial_tlist = []
        for p in range(4):
            if p > 0:
                stim_t = trial_tlist[p-1] + isi_dur + randng.uniform(low =-jitter,high=jitter)
            else:
                stim_t = isi_dur + randng.uniform(low =-jitter,high=jitter)
            trial_tlist.append(stim_t)
        block_tlist.append(trial_tlist)
    return block_tlist

def generateBlockLevels(nblocks: int):
    """ 
    Generate list with half half zeros and another half ones for any block level parameter
    The attention conditions, which direction is odd or which angle is odd
    Parameters:
        nblocks (int): amount of blocks
    Returns:
        cond_list (ndarray): array of length nblocks with equal amounts 1's and 0's 
    """
    # Make a list with normal trials (0) and a list with target trials (1)
    half_block = int(np.ceil(nblocks/2))
    cond_1 = np.full(half_block,0,dtype=int)
    cond_2 = np.full(half_block,1,dtype=int)
    full_cond = np.concatenate([cond_1,cond_2]) 
    np.random.shuffle(full_cond)
    cond_list = full_cond[:nblocks]
    return cond_list

def generatePredictionList(tr_block, n_odd, odd):
    """ 
    Generate pseudo randomized properties of trial direction or stimulus angle on a block level
    Parameters:
        tr_block (int):number of trials per block
        n_odd (int): number of odd per block
        odd (int): indicate which direction or angle is odd 0 clockwise/left and 1 anticlockwise/right
    Returns:
        predicition_arr (ndarray): array with length tr_block with 0's and 1's for prediciton condition 
    """
    # Check if there are correct amount of odds
    if n_odd > tr_block/2:
        raise ValueError("Invalid input parameters")

    prediction_arr = np.zeros(tr_block , dtype=int)
    possible_positions = np.arange(tr_block)
    # Randomly select positions for 1's, ensuring no two are consecutive
    selected_positions = []
    while len(selected_positions) < n_odd:
        pos = np.random.choice(possible_positions)
        selected_positions.append(pos)
        # Remove the selected position and its adjacent positions to prevent consecutive 1's
        possible_positions = possible_positions[(possible_positions < pos - 1) | (possible_positions > pos + 1)]
    prediction_arr[selected_positions] = 1
    
    # Flip the 0's and 1's depending on which is odd
    if not odd:
        prediction_arr = prediction_arr ^ 1
        
    return prediction_arr


In [4]:
timings = generateTrialTimings(tr_block=tr_block,isi_dur=isi_duration,jitter=stim_onset_jitter,randng=rng)
targets = generateTargetTrials(tr_block=tr_block,ntarget=n_target)
# Generate block-level conditions
block_condition = generateBlockLevels(3)
# Participant specific values
rotodd_map = ['clockwise','anticlockwise']
angodd_map = ['left','right']
key_map = [['k','d'],['d','k']]

subject_code = [0,0,1,0] 
rotation_odd = rotodd_map[subject_code[0]] # index 0 idicate which direction is odd
angle_odd = angodd_map[subject_code[1]] # index 1 indicates which angle is odd
rotation_keys = key_map[subject_code[2]] # index 2 indicates which keys are paired to the directions
angle_keys = key_map[subject_code[3]] # index 3 indicates which keys are paired to the angles

key_mappings = [rotation_keys,angle_keys]
key_map_dict = {rotodd_map[0]:rotation_keys[0],rotodd_map[1]:rotation_keys[1],
                angodd_map[0]:angle_keys[0],angodd_map[1]:angle_keys[1]}


NameError: name 'tr_block' is not defined

In [ ]:
def evaluateResponse(stim_times,target,cond,direction,angles,keys_mapped,key):
    """
    Evaluate the response to the trial
    Parameters:
        stim_times (list of floats): the times a stimulus apeared i.r.t. the trial start
        target (int): if target trial and which (0 or 2-3)
        cond (int): which condition 0 rotation or 1 angles
        direction (int): rotation direction 0 clockwise or 1 anticlockwise
        angles (list): list of angles for every stim in the trial 0 left or 1 right
        keys_mapped (list of list): list of key mapping for the subject [rotation keys (list of k and d),angle keys (list of k and d)] 
        key (list of tuples):
    Returns:
        acc (str): gives the accuracy of the response
        rt (float or None): gives reaction times if subject reacted
    """
    if not key: # If no key was pressed
        if target:
            acc = 'miss', 
        else:
            acc = 'correct'
        rt = None 
        return acc, rt
    else: # If key was pressed
        rt = key[0][1] - stim_times[target-1]
        if target:
            acc = 'wrong'
            if rt < 0:
                acc = 'false_fire'
                return acc, rt
            if rt > 0.9:
                acc = 'too_slow'
                return acc, rt
            if not cond:
                if key[0][0] == keys_mapped[0][direction]:
                    acc = 'correct'
            else:
                if key[0][0] == keys_mapped[1][angles[target]]:
                    acc = 'correct'
            return acc, rt
        else:
            acc = 'false_fire'
            return acc, rt
                

In [5]:
def generateAngleList(angle_pred_list, odd):
    """ 
    Generate list for the angles depending on prediction conditions generated by previous function
    Parameters:
        angle_pred_list (ndarray): prediction list for only angles
        odd (int): indicate which direction or angle is odd 0 clockwise/left and 1 anticlockwise/right
    Returns:
        angle_list (list of list): list to set the angle of each stimulus per trial in a block
    """
    # Possible sets for every trial (4 stimuli)
    odd_options = [[odd,odd,odd,odd]] # depends on which number indicates odd
    reg_options = [1 if x == 0 else 0 for x in [odd,odd,odd,odd]]
    # Init list
    angle_list = []
    # Iterate over trial level prediciton list
    for i in angle_pred_list:
        if i == odd: # If odd add one of the odd options
            angle_list.append(odd_options)
        else: # for regular add all the same
            angle_list.append(reg_options)
    return angle_list

In [6]:
odd = 1
predi = generatePredictionList(tr_block=n_trials,n_odd=n_odd,odd=odd)
stim_angles = generateAngleList(predi,odd)

In [7]:
stim_angles[0]


[[1, 1, 1, 1]]

In [133]:
# Generate trial-level conditions per block
block_rot_pred = []
block_angle_pred = []
block_stimangle_pred = []
trial_starts = []
target_trials = []
trial_timings = []
# Experiment wide lists
trial_list = [{'trial_index':i} for i in range(n_blocks*n_trials)] # for trialhandler
recalibrated = np.zeros(n_blocks*n_trials)

subject_code = [0,1,1,0] 

for i in range(n_blocks):
    block_rot_pred.append(generatePredictionList(tr_block=n_trials,n_odd=n_odd,
                                            odd=subject_code[0]))
    angle_list = generatePredictionList(tr_block=n_trials,n_odd=n_odd,
                                            odd=subject_code[1])
    stimangle_list = generateAngleList(angle_pred_list=angle_list,odd=subject_code[1])
    block_angle_pred.append(angle_list)
    block_stimangle_pred.append(stimangle_list)
    target_trials.append(generateTargetTrials(tr_block=n_trials,ntarget=n_target))
    trial_timings.append(generateTrialTimings(tr_block=n_trials,isi_dur=isi_duration,
                                        jitter=stim_onset_jitter))


In [134]:
for id in range(n_blocks):
    for i in range(n_trials):
        rzzze = block_stimangle_pred[id][i]
        print(rzzze)

[0, 1, 1, 0]
[0, 0, 0, 0]
[0, 0, 0, 0]
[0, 0, 0, 0]
[0, 0, 0, 0]
[0, 1, 0, 1]
[0, 0, 0, 0]
[0, 1, 0, 1]
[0, 0, 0, 0]
[0, 0, 0, 0]
[1, 1, 0, 0]
[0, 0, 0, 0]
[0, 0, 0, 0]
[0, 0, 0, 0]
[0, 1, 1, 0]
[0, 0, 0, 0]
[0, 0, 0, 0]
[0, 0, 0, 0]
[0, 1, 0, 1]
[0, 0, 0, 0]
[0, 1, 0, 1]
[0, 0, 0, 0]
[0, 0, 0, 0]
[0, 0, 0, 0]
[0, 0, 0, 0]
[0, 0, 0, 0]
[0, 0, 0, 0]
[0, 0, 0, 0]
[0, 0, 0, 0]
[0, 0, 0, 0]
[0, 1, 0, 1]
[0, 0, 0, 0]
[0, 0, 0, 0]
[0, 1, 0, 1]
[0, 0, 0, 0]
[0, 0, 0, 0]
[0, 0, 0, 0]
[0, 0, 0, 0]
[0, 0, 0, 0]
[0, 0, 0, 0]
[1, 1, 0, 0]
[0, 0, 0, 0]
[0, 0, 0, 0]
[0, 0, 0, 0]
[0, 0, 0, 0]
[0, 0, 0, 0]
[0, 1, 0, 1]
[0, 0, 0, 0]
[0, 0, 0, 0]
[0, 1, 0, 1]
[0, 0, 0, 0]
[0, 0, 0, 0]
[0, 1, 0, 1]
[0, 0, 0, 0]
[0, 0, 0, 0]
[0, 1, 0, 1]
[0, 0, 0, 0]
[1, 1, 0, 0]
[0, 0, 0, 0]
[1, 1, 0, 0]
[0, 0, 0, 0]
[0, 0, 0, 0]
[0, 0, 0, 0]
[0, 0, 0, 0]
[0, 0, 0, 0]
[1, 1, 0, 0]
[0, 0, 0, 0]
[0, 0, 0, 0]
[0, 0, 0, 0]
[0, 0, 0, 0]
[0, 0, 0, 0]
[0, 1, 0, 1]
[0, 0, 0, 0]
[0, 0, 0, 0]
[0, 0, 0, 0]
[0, 0, 0, 0]
[0, 0, 0, 0]

In [136]:
full_list = np.zeros(n_trials-8,dtype=int)
c_list = np.ones(8*3,dtype=int) # make a longer list and cut it short otherwise it won't work with ntarget <3
# Make correct length
np.random.shuffle(c_list)
c_list = c_list[:8]

# Make one list
full_list = np.append(full_list,c_list)
# Shuffle
np.random.shuffle(full_list)

In [2]:
def participantCounterBalance(participant_number,conds = 4):
    """ 
    Assign particpant to conditions such as key-mappings and odd vs regular direction/angles
    Parameters:
        participant_number (int):identifier that serves as unique key for assignment
        conds (int): number of conditions
    Returns:
        counter_code (ndarray): array with length conds with a code for every participant 
    """
    counter_code = np.empty(conds, dtype=int)
    for i in range(conds):
        counter_code[i] = (participant_number // (i+1)) % 2

    return counter_code

In [8]:
for i in range(40):
    print(participantCounterBalance(i,conds=4))

[0 0 0 0]
[1 0 0 0]
[0 1 0 0]
[1 1 1 0]
[0 0 1 1]
[1 0 1 1]
[0 1 0 1]
[1 1 0 1]
[0 0 0 0]
[1 0 1 0]
[0 1 1 0]
[1 1 1 0]
[0 0 0 1]
[1 0 0 1]
[0 1 0 1]
[1 1 1 1]
[0 0 1 0]
[1 0 1 0]
[0 1 0 0]
[1 1 0 0]
[0 0 0 1]
[1 0 1 1]
[0 1 1 1]
[1 1 1 1]
[0 0 0 0]
[1 0 0 0]
[0 1 0 0]
[1 1 1 0]
[0 0 1 1]
[1 0 1 1]
[0 1 0 1]
[1 1 0 1]
[0 0 0 0]
[1 0 1 0]
[0 1 1 0]
[1 1 1 0]
[0 0 0 1]
[1 0 0 1]
[0 1 0 1]
[1 1 1 1]
